# V5 one-time raw input migration
Copies only original source folders, verifies SHA-256, never deletes old data. Requires sufficient Drive quota. This is not NB00 or a full pipeline release. Run once from your own Colab account. Satellite inputs and reference contracts are separate outstanding dependencies.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import shutil, hashlib, json
src=Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final/Data')
root=src.parent/'V5'
dst=root/'Inputs'
qa=root/'QA_and_Manifests'
qa.mkdir(parents=True,exist_ok=True)
dst.mkdir(parents=True,exist_ok=True)
assert src.is_dir() and src.resolve()!=dst.resolve()
folders=('Ortho_RGB','MS','DSM','Annotations','Field','LAS','Patrick')
def digest(p):
    h=hashlib.sha256()
    with p.open('rb') as f:
        for b in iter(lambda:f.read(8*1024*1024),b''):h.update(b)
    return h.hexdigest()
files=[]
for folder in folders:
    assert (src/folder).is_dir(), folder
    for p in sorted((src/folder).rglob('*')):
        if p.is_symlink():raise RuntimeError(f'Linked source: {p}')
        if p.is_file():files.append(p)
assert files
need=sum(p.stat().st_size for p in files if not (dst/p.relative_to(src)).exists())
print('Additional bytes required:',need,'Reported free bytes:',shutil.disk_usage(dst).free)
assert need<=shutil.disk_usage(dst).free
entries=[]
for p in files:
    rel=p.relative_to(src)
    q=dst/rel
    q.parent.mkdir(parents=True,exist_ok=True)
    h=digest(p)
    if q.exists():
        assert q.is_file() and not q.is_symlink() and q.stat().st_size==p.stat().st_size and digest(q)==h, f'Existing destination mismatch: {q}'
    else:
        tmp=q.with_name(q.name+'.copying')
        shutil.copyfile(p,tmp)
        assert tmp.stat().st_size==p.stat().st_size and digest(tmp)==h
        tmp.replace(q)
    entries.append(dict(relative_path=rel.as_posix(),bytes=p.stat().st_size,sha256=h,provenance='Original Final/Data/'+rel.as_posix()))
    print('Verified',rel,flush=True)
observed={p.relative_to(dst).as_posix() for p in dst.rglob('*') if p.is_file()}
assert observed=={e['relative_path'] for e in entries}, 'Unexpected files in Inputs; inspect before freezing'
manifest=qa/'input_manifest.json'
obj=dict(schema='V5_raw_input_inventory_1',files=entries)
if manifest.exists():
    assert json.loads(manifest.read_text())==obj, 'Existing frozen manifest differs'
else:
    tmp=manifest.with_suffix('.tmp')
    tmp.write_text(json.dumps(obj,indent=2))
    tmp.replace(manifest)
print('Raw inputs copied and byte-verified:',len(entries),'files. Original files retained.')

